# Project 1 -- Building a Search Engine from Scratch
**Course:** Multimedia Information Retrieval & Computer Vision (University of Pisa, A.Y. 2026/2027)

---
## Notebook structure
1. Set-up: package installation with pinned versions (pip install package==x.y.z), Google
Drive mounting, and one configuration cell holding all paths and flags (stemming, stop-word
removal, buffer size, BM25 parameters, ...). Print the runtime characteristics: Python version,
number of cpu cores, ram.
2. Data: download of the collection, queries, and judgements.
3. Implementation: document processing, indexing, query processing, scoring, and extensions. Longer code may be written to modules with %%writefile from within the notebook and im-ported.
4. Tests.
5. Indexing runs, printing time, peak memory, and index size.
6. Evaluation and experiments, producing every table and plot of the report.
7. Report.

---
## 1. Set-up
This section installs the allowed packages with pinned versions, mounts Google Drive for persistent storage, defines the global configuration cell, and logs the hardware and runtime specifications.

In [ ]:
# Package installation with pinned versions
!pip install -q \
    ir_datasets==0.5.6 \
    ir_measures==0.3.3 \
    PyStemmer==2.2.0.1 \
    numpy==1.26.4 \
    scipy==1.13.1 \
    pandas==2.2.2 \
    tqdm==4.66.4 \
    psutil==5.9.8 \
    matplotlib==3.8.4 \
    seaborn==0.13.2 \
    cython==3.0.10

In [ ]:
# Environment imports, Google Drive mount, and runtime inspection
# 1.2 Environment imports, Google Drive mount, and runtime inspection
import os
import sys
import time
import shutil
import platform
import resource
from dataclasses import dataclass
from pathlib import Path
import psutil

def get_peak_rss_mb() -> float:
    """Returns peak resident set size (RSS) in MB using resource.getrusage"""
    # On Linux, ru_maxrss is in kilobytes; on macOS, it is in bytes
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    divisor = (1024 ** 2) if sys.platform == "darwin" else 1024
    return usage / divisor

# Mount Google Drive if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive                                      # TODO
    drive.mount("/content/drive", force_remount=False)                  # <----------- WE NEED TO CREATE THE FOLDER --------------
    BASE_LOCAL_DIR = Path("/content")                                   # <-------------------------------------------------------
    BASE_DRIVE_DIR = Path("/content/drive/MyDrive/MIRCV_Project1")      # <-------------------------------------------------------
else:
    # Local fallback if testing outside Colab
    BASE_LOCAL_DIR = Path("./local_workspace")                          # <-------------------------------------------------------
    BASE_DRIVE_DIR = Path("./drive_workspace")                          # <-------------------------------------------------------

BASE_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
BASE_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

def print_runtime_characteristics() -> None:
    """Prints Python version, CPU cores, RAM, and disk space"""
    vm = psutil.virtual_memory()
    local_disk = shutil.disk_usage(BASE_LOCAL_DIR)
    drive_disk = shutil.disk_usage(BASE_DRIVE_DIR)
    
    print("=" * 60)
    print("RUNTIME CHARACTERISTICS")
    print("=" * 60)
    print(f"Platform       : {platform.platform()}")
    print(f"Python Version : {sys.version.split()[0]} ({platform.python_implementation()})")
    print(f"CPU Cores      : {os.cpu_count()} logical cores ({psutil.cpu_count(logical=False)} physical)")
    print(f"Total RAM      : {vm.total / (1024**3):.2f} GB (Available: {vm.available / (1024**3):.2f} GB)")
    print(f"Process Peak   : {get_peak_rss_mb():.2f} MB")
    print(f"Local Disk     : {local_disk.free / (1024**3):.2f} GB free / {local_disk.total / (1024**3):.2f} GB total ({BASE_LOCAL_DIR})")
    print(f"Drive Disk     : {drive_disk.free / (1024**3):.2f} GB free / {drive_disk.total / (1024**3):.2f} GB total ({BASE_DRIVE_DIR})")
    print("=" * 60)

print_runtime_characteristics()

In [ ]:
# Global Configuration Cell
@dataclass
class EngineConfig:
    # --- Execution and dataset mode ---
    use_dev_subset: bool = False          # True = use first 100,000 passages for fast debugging; False = full 8.8M collection
    dev_subset_size: int = 100_000        # Number of lines in the development subset
    force_reindex: bool = False           # If False, skips indexing when the final index already exists on Drive
    
    # --- Document and query text processing flags ---
    use_stopwords: bool = True            # Switchable stop-word removal flag
    use_stemming: bool = True             # Switchable Snowball/Porter stemming flag
    unicode_normalisation: str = "NFKC"   # Optional Unicode canonicalisation ("NFKC" or None)
    
    # --- Memory-bounded SPIMI indexing parameters ---
    # Max number of (docid, tf) postings held in RAM before flushing an intermediate run to disk.
    max_buffer_postings: int = 15_000_000 
    cleanup_intermediate_runs: bool = True # Delete temporary block runs once final index is merged (saves Drive/disk space)
    
    # --- Query processing and scoring parameters ---
    bm25_k1: float = 0.9                  # Mandatory default k1 = 0.9
    bm25_b: float = 0.4                   # Mandatory default b = 0.4
    default_top_k: int = 1000             # Default number of retrieved documents for TREC runs
    
    # --- Extensions configuration --- TODO: choose Extensions
    # E1: "none" (raw uint32), "vbyte" (d-gaps + Variable-Byte), "text" (human-readable debug format)
    compression_format: str = "none"      
    # E2: Skip pointers block size (0 = disabled, >0 = block size or sqrt(df) if -1)
    skip_block_size: int = 128            
    # E2: Dynamic pruning strategy for disjunctive queries: "none", "maxscore", or "wand"
    pruning_strategy: str = "none"        
    # E3: Traversal strategy: "daat" (Document-At-A-Time) or "taat" (Term-At-A-Time)
    traversal_strategy: str = "daat"      
    
    # --- Storage paths ---
    local_dir: Path = BASE_LOCAL_DIR
    drive_dir: Path = BASE_DRIVE_DIR

    @property
    def collection_archive_path(self) -> Path:
        return self.local_dir / "collection.tar.gz"

    @property
    def full_collection_tsv_path(self) -> Path:
        return self.local_dir / "collection.tsv"

    @property
    def dev_collection_tsv_path(self) -> Path:
        return self.local_dir / f"collection_dev_{self.dev_subset_size}.tsv"

    @property
    def active_collection_path(self) -> Path:
        return self.dev_collection_tsv_path if self.use_dev_subset else self.full_collection_tsv_path

    def get_index_name(self, stopwords: bool = None, stemming: bool = None, compression: str = None) -> str:
        sw = self.use_stopwords if stopwords is None else stopwords
        st = self.use_stemming if stemming is None else stemming
        comp = self.compression_format if compression is None else compression
        scope = f"dev{self.dev_subset_size}" if self.use_dev_subset else "full"
        return f"index_{scope}_sw{int(sw)}_stem{int(st)}_{comp}"

    def get_drive_index_dir(self, **kwargs) -> Path:
        return self.drive_dir / "indices" / self.get_index_name(**kwargs)

    def get_local_index_dir(self, **kwargs) -> Path:
        return self.local_dir / "indices" / self.get_index_name(**kwargs)

    @property
    def runs_dir(self) -> Path:
        path = self.drive_dir / "runs"
        path.mkdir(parents=True, exist_ok=True)
        return path

CONFIG = EngineConfig()
print(f"Active configuration -> Index tag: {CONFIG.get_index_name()}")
print(f"Active collection    -> {CONFIG.active_collection_path}")

---
## 2. Data
This section downloads and extracts the MS MARCO passage collection (`collection.tsv`, 8,841,823 passages) directly onto the local runtime disk (`/content`) to preserve Google Drive quota and maximise I/O throughput. It also creates the 100,000-passage development subset and loads the three standard evaluation query sets and relevance judgements via `ir_datasets`.

In [ ]:
# Download and extract MS MARCO passage collection (idempotent)
import tarfile
from itertools import islice

# Official MS MARCO Passage Ranking collection URL (with mirror fallback)
MSMARCO_URLS = [
    "https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz",
    "https://msmarco.blob.core.windows.net/msmarcoranking/collection.tar.gz"
]

def prepare_collection(cfg: EngineConfig) -> None:
    """Downloads collection.tar.gz, extracts collection.tsv, and builds the 100k dev subset."""
    # Extract or download full collection.tsv
    if cfg.full_collection_tsv_path.exists() and cfg.full_collection_tsv_path.stat().st_size > 2_900_000_000:
        print(f"Full collection already extracted at {cfg.full_collection_tsv_path} "
              f"({cfg.full_collection_tsv_path.stat().st_size / (1024**3):.2f} GB)")
    else:
        if not cfg.collection_archive_path.exists():
            print("Downloading collection.tar.gz (1 GB)...")
            downloaded = False
            for url in MSMARCO_URLS:
                exit_code = os.system(f'wget -q --show-progress -O "{cfg.collection_archive_path}" "{url}"')
                if exit_code == 0 and cfg.collection_archive_path.exists() and cfg.collection_archive_path.stat().st_size > 900_000_000:
                    downloaded = True
                    break
                print(f"Warning: download from {url} failed, trying fallback...")
            if not downloaded:
                raise RuntimeError("Failed to download collection.tar.gz from all mirrors.")
        
        print("Extracting collection.tsv (3 GB) to local disk...")
        t0 = time.perf_counter()
        with tarfile.open(cfg.collection_archive_path, "r:gz") as tar:
            tar.extract("collection.tsv", path=cfg.local_dir)
        print(f"Extraction completed in {time.perf_counter() - t0:.1f}s.")
        
        # Remove the compressed archive after extraction to free 1 GB on /content
        if cfg.collection_archive_path.exists():
            cfg.collection_archive_path.unlink()
            print("Removed temporary collection.tar.gz to free local disk space.")

    # 2. Build the 100,000-line development subset
    if cfg.dev_collection_tsv_path.exists():
        print(f"Development subset already exists at {cfg.dev_collection_tsv_path}")
    else:
        print(f"Creating development subset ({cfg.dev_subset_size:,} lines)...")
        with open(cfg.full_collection_tsv_path, "rb") as src, \
             open(cfg.dev_collection_tsv_path, "wb") as dst:
            for line in islice(src, cfg.dev_subset_size):
                dst.write(line)
        print(f"Development subset written to {cfg.dev_collection_tsv_path} "
              f"({cfg.dev_collection_tsv_path.stat().st_size / (1024**2):.2f} MB)")

prepare_collection(CONFIG)

# Preview the first 3 lines of the collection
print("\nSample passages from collection.tsv:")
with open(CONFIG.active_collection_path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(3):
        pid_str, text_str = f.readline().rstrip("\r\n").split("\t", 1)
        print(f"  [Line {i}] pid={pid_str} | text={text_str[:90]}...")

In [ ]:
# Load test queries and relevance judgements (qrels) via ir_datasets
import ir_datasets
from ir_measures import Qrel

QUERY_SETS_SPECS = {
    "dl19": {
        "ir_id": "msmarco-passage/trec-dl-2019/judged",
        "expected_queries": 43,
        "description": "TREC DL 2019 (graded, 0-3)"
    },
    "dl20": {
        "ir_id": "msmarco-passage/trec-dl-2020/judged",
        "expected_queries": 54,
        "description": "TREC DL 2020 (graded, 0-3)"
    },
    "dev_small": {
        "ir_id": "msmarco-passage/dev/small",
        "expected_queries": 6980,
        "description": "MS MARCO dev (small) (binary)"
    }
}

@dataclass
class EvaluationDataset:
    name: str
    ir_id: str
    description: str
    queries: dict[str, str]          # qid -> raw query text
    qrels: list[Qrel]                # list of ir_measures.Qrel(query_id, doc_id, relevance)
    judged_qids: list[str]           # sorted list of judged query IDs for exact averaging & paired t-test

def load_evaluation_datasets() -> dict[str, EvaluationDataset]:
    """Loads all three benchmark query sets and their relevance judgements into memory."""
    loaded = {}
    print(f"{'Dataset Key':<12} | {'ir_datasets Identifier':<36} | {'Queries':>8} | {'Judged QIDs':>11} | {'Qrels':>8}")
    print("-" * 88)
    
    for key, spec in QUERY_SETS_SPECS.items():
        ds = ir_datasets.load(spec["ir_id"])
        
        # Load queries preserving order
        queries = {str(q.query_id): str(q.text) for q in ds.queries_iter()}
        
        # Load relevance judgements in ir_measures format
        qrels = [
            Qrel(query_id=str(r.query_id), doc_id=str(r.doc_id), relevance=int(r.relevance))
            for r in ds.qrels_iter()
        ]
        
        # Identify all unique judged query IDs (crucial for Section 7.2 zero-result handling)
        judged_qids = sorted({q.query_id for q in qrels})
        
        # Sanity check against Project 1 Specifications (Section 4.2)
        assert len(queries) == spec["expected_queries"], (
            f"Expected {spec['expected_queries']} queries for {key}, got {len(queries)}"
        )
        
        loaded[key] = EvaluationDataset(
            name=key,
            ir_id=spec["ir_id"],
            description=spec["description"],
            queries=queries,
            qrels=qrels,
            judged_qids=judged_qids
        )
        print(f"{key:<12} | {spec['ir_id']:<36} | {len(queries):>8,d} | {len(judged_qids):>11,d} | {len(qrels):>8,d}")
        
    return loaded

BENCHMARKS = load_evaluation_datasets()

---
## 3. Implementation
The retrieval system is decomposed into four cohesive modules:
1. text_processor: tokenisation, normalisation, switchable stop-word removal, switchable Snowball stemming with token-level memoization, and fault-tolerant TSV line parsing.
2. fast_codecs: VByte encoder/decoder with gap encoding (d-gaps) and posting-list scoring kernels.
3. postings_io: unified posting-list storage and cursor interface supporting uncompressed `uint32` binary, VByte compressed binary, human readable text format, and skip pointer acceleration.
4. indexer: memory bounded and single pass In-Memory Indexing (SPIMI) with restartable intermediate block runs and k-way merge.
5. query_processor: DAAT and TAAT query evaluation for conjunctive (AND) and disjunctive (OR) queries, TF-IDF and BM25 scoring, and safe dynamic pruning (MaxScore and WAND).

In [ ]:
%%writefile text_processor.py
import re
import unicodedata
import Stemmer

# Standard English stopword list from Snowball / NLTK.
# Source: Porter, M. F. (2001) (https://snowballstem.org/)
ENGLISH_STOP_WORDS = frozenset({
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she",
    "her", "hers", "herself", "it", "its", "itself", "they", "them", "their",
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that",
    "these", "those", "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an",
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of",
    "at", "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down",
    "in", "out", "on", "off", "over", "under", "again", "further", "then",
    "once", "here", "there", "when", "where", "why", "how", "all", "any",
    "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor",
    "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can",
    "will", "just", "don", "should", "now", "d", "ll", "m", "o", "re", "ve",
    "y", "ain", "aren", "couldn", "didn", "doesn", "hadn", "hasn", "haven",
    "isn", "ma", "mightn", "mustn", "needn", "shan", "shouldn", "wasn", "weren",
    "won", "wouldn"
})

# Matches any sequence of characters that are not alphanumeric
NON_ALNUM_RE = re.compile(r"[^\w]+|_+")


class TextProcessor:
    """
    Text processing pipeline for documents and queries
    Steps: (0) Optional Unicode NFKC -> (1) Lowercase -> (2) Punctuation removal
           -> (3) Whitespace tokenisation -> (4) Stop-word removal -> (5) Stemming
    """
    def __init__(
        self,
        use_stopwords: bool = True,
        use_stemming: bool = True,
        unicode_normalisation: str | None = "NFKC"
    ) -> None:
        self.use_stopwords = use_stopwords
        self.use_stemming = use_stemming
        self.unicode_normalisation = unicode_normalisation
        self.stop_words = ENGLISH_STOP_WORDS if use_stopwords else frozenset()
        self.stemmer = Stemmer.Stemmer("english") if use_stemming else None
        # Memoization cache for stemming: maps raw token -> stemmed term
        self._stem_cache: dict[str, str] = {}

    def process(self, text: str) -> list[str]:
        if not text:
            return []
        if self.unicode_normalisation:
            text = unicodedata.normalize(self.unicode_normalisation, text)
        
        # Case folding, non-alphanumeric replacement with space, whitespace split
        tokens = NON_ALNUM_RE.sub(" ", text.lower()).split()
        if not tokens:
            return []

        # Stopwords removal
        if self.use_stopwords:
            tokens = [t for t in tokens if t not in self.stop_words]
            if not tokens:
                return []

        # Stemming with token-level memoization
        if self.use_stemming and self.stemmer is not None:
            cache = self._stem_cache
            stem_word = self.stemmer.stemWord
            out: list[str] = []
            for t in tokens:
                stemmed = cache.get(t)
                if stemmed is None:
                    stemmed = stem_word(t)
                    cache[t] = stemmed
                out.append(stemmed)
            return out

        return tokens


def parse_tsv_line(raw_line: str) -> tuple[int | None, str | None]:
    """
    Parses a single 'pid \t <text> \n' line from collection.tsv (Section 4.1).
    Returns (None, None) if the line has no tab or if pid is not a valid integer.
    """
    tab_pos = raw_line.find("\t")
    if tab_pos == -1:
        return None, None
    pid_str = raw_line[:tab_pos]
    try:
        pid = int(pid_str)
    except ValueError:
        return None, None
    text = raw_line[tab_pos + 1:].rstrip("\r\n")
    return pid, text

In [ ]:
# Load Cython extension to compile VByte and d-gap loops
%load_ext Cython

In [ ]:
%%cython
import numpy as np
cimport numpy as cnp
from libc.stdint cimport uint32_t, uint8_t

cnp.import_array()


cpdef bytes vbyte_encode_array(const uint32_t[:] values, bint use_dgaps):
    """
    Encodes a uint32 array using Variable-Byte (VByte) encoding.
    If use_dgaps=True, encodes strictly positive gaps:
    g_0 = values[0] + 1 (since docids start at 0), and g_i = values[i] - values[i-1].
    """
    cdef Py_ssize_t n = values.shape[0]
    if n == 0:
        return b""
    
    # Maximum 5 bytes per uint32 integer
    cdef cnp.ndarray[uint8_t, ndim=1] out = np.empty(n * 5, dtype=np.uint8)
    cdef uint8_t* out_ptr = &out[0]
    cdef Py_ssize_t out_idx = 0
    cdef Py_ssize_t i
    cdef uint32_t val, prev = 0

    for i in range(n):
        if use_dgaps:
            if i == 0:
                val = values[0] + 1
            else:
                val = values[i] - prev
            prev = values[i]
        else:
            val = values[i]

        while val >= 128:
            out_ptr[out_idx] = <uint8_t>((val & 0x7F) | 0x80)
            out_idx += 1
            val >>= 7
        out_ptr[out_idx] = <uint8_t>(val & 0x7F)
        out_idx += 1

    return out[:out_idx].tobytes()


cpdef tuple vbyte_decode_lists(const uint8_t[:] buf, Py_ssize_t offset, int df):
    """
    Decodes both the d-gap compressed docids (length df) and VByte compressed tfs (length df)
    starting at offset inside buf.
    Returns (docids_ndarray, tfs_ndarray).
    """
    cdef cnp.ndarray[uint32_t, ndim=1] docids = np.empty(df, dtype=np.uint32)
    cdef cnp.ndarray[uint32_t, ndim=1] tfs = np.empty(df, dtype=np.uint32)
    cdef uint32_t* doc_ptr = &docids[0]
    cdef uint32_t* tf_ptr = &tfs[0]

    cdef Py_ssize_t pos = offset
    cdef int i
    cdef uint32_t x, shift, b
    cdef uint32_t acc = 0

    # Decode d-gaps and reconstruct original sorted docids via prefix sum
    for i in range(df):
        x = 0
        shift = 0
        while True:
            b = buf[pos]
            pos += 1
            x |= (b & 0x7F) << shift
            if b < 128:
                break
            shift += 7
        if i == 0:
            acc = x - 1
        else:
            acc += x
        doc_ptr[i] = acc

    # Decode raw term frequencies (tf >= 1)
    for i in range(df):
        x = 0
        shift = 0
        while True:
            b = buf[pos]
            pos += 1
            x |= (b & 0x7F) << shift
            if b < 128:
                break
            shift += 7
        tf_ptr[i] = x

    return docids, tfs

In [ ]:
%%writefile postings_io.py
import math
import numpy as np
from fast_codecs import vbyte_encode_array, vbyte_decode_lists

SENTINEL_DOCID = 0xFFFFFFFF  # Infinity sentinel (2^32 - 1) when cursor is exhausted


def write_posting_list(
    f_out,
    term: str,
    docids: np.ndarray,
    tfs: np.ndarray,
    compression_format: str = "none"
) -> tuple[int, int]:
    """
    Writes a single term's posting list (docids, tfs) to `f_out`.
    Returns (byte_offset, byte_length).
    Supported formats:
      - "none"  : raw uint32 docids followed by raw uint32 tfs
      - "vbyte" : d-gap VByte docids followed by VByte tfs
      - "text"  : human-readable text format for debugging
    """
    offset = f_out.tell()

    if compression_format == "none":
        f_out.write(docids.tobytes())
        f_out.write(tfs.tobytes())
    elif compression_format == "vbyte":
        enc_docs = vbyte_encode_array(docids, True)
        enc_tfs = vbyte_encode_array(tfs, False)
        f_out.write(enc_docs)
        f_out.write(enc_tfs)
    elif compression_format == "text":
        pairs_str = " ".join(f"{int(d)}:{int(tf)}" for d, tf in zip(docids, tfs))
        line_bytes = f"{term}\t{pairs_str}\n".encode("utf-8")
        f_out.write(line_bytes)
    else:
        raise ValueError(f"Unsupported compression_format: {compression_format}")

    byte_length = f_out.tell() - offset
    return offset, byte_length


class PostingListIterator:
    """
    Cursor over a posting list read on-demand from disk/mmap.
    Supports sequential traversal and sub-linear skipping via next_geq(target)
    using skip pointers with configurable block step.
    """
    __slots__ = ("docids", "tfs", "n", "pos", "skip_step", "ub_score", "idf")

    def __init__(
        self,
        docids: np.ndarray,
        tfs: np.ndarray,
        skip_block_size: int = 128,
        ub_score: float = 0.0,
        idf: float = 0.0
    ) -> None:
        self.docids = docids
        self.tfs = tfs
        self.n = int(docids.shape[0])
        self.pos = 0
        self.ub_score = ub_score
        self.idf = idf

        # Configure skip pointer step (optimal s = floor(sqrt(n)))
        if skip_block_size == -1:
            self.skip_step = max(1, math.isqrt(self.n))
        elif skip_block_size > 0:
            self.skip_step = skip_block_size
        else:
            self.skip_step = 0  # Skipping disabled (linear scan only)

    @property
    def docid(self) -> int:
        if self.pos >= self.n:
            return SENTINEL_DOCID
        return int(self.docids[self.pos])

    @property
    def tf(self) -> int:
        return int(self.tfs[self.pos])

    def advance(self) -> int:
        """Advances cursor by 1 position and returns the new docid (or SENTINEL_DOCID)."""
        self.pos += 1
        if self.pos >= self.n:
            return SENTINEL_DOCID
        return int(self.docids[self.pos])

    def next_geq(self, target: int) -> int:
        """
        Nextgeq: advances the cursor to the smallest entry d >= target
        and returns d, or SENTINEL_DOCID if no such entry exists.
        Uses skip pointers when self.skip_step > 1.
        """
        pos = self.pos
        n = self.n
        if pos >= n:
            return SENTINEL_DOCID

        docs = self.docids
        curr = int(docs[pos])
        if curr >= target:
            return curr

        step = self.skip_step
        # Jump over blocks of size step using skip pointers
        if step > 1 and (n - pos) > step:
            # Align to skip boundaries or jump by fixed step
            while pos + step < n and int(docs[pos + step]) <= target:
                pos += step

        # Linear scan within the landed block
        while pos < n and int(docs[pos]) < target:
            pos += 1

        self.pos = pos
        if pos >= n:
            return SENTINEL_DOCID
        return int(docs[pos])


def load_posting_list_from_mmap(
    mmap_buf,
    offset: int,
    byte_length: int,
    df: int,
    compression_format: str = "none",
    skip_block_size: int = 128,
    ub_score: float = 0.0,
    idf: float = 0.0
) -> PostingListIterator:
    """
    Constructs a PostingListIterator for a single query term by reading only its byte range
    from the memory-mapped postings file on disk.
    """
    if compression_format == "none":
        # Zero copy numpy views directly over the mmap slice
        docids = np.frombuffer(mmap_buf, dtype=np.uint32, count=df, offset=offset)
        tfs = np.frombuffer(mmap_buf, dtype=np.uint32, count=df, offset=offset + df * 4)
    elif compression_format == "vbyte":
        mv = memoryview(mmap_buf)
        docids, tfs = vbyte_decode_lists(mv, offset, df)
    elif compression_format == "text":
        raw_line = mmap_buf[offset:offset + byte_length].decode("utf-8")
        _, pairs_part = raw_line.rstrip("\r\n").split("\t", 1)
        docids = np.empty(df, dtype=np.uint32)
        tfs = np.empty(df, dtype=np.uint32)
        for i, token in enumerate(pairs_part.split(" ")):
            d_str, tf_str = token.split(":")
            docids[i] = int(d_str)
            tfs[i] = int(tf_str)
    else:
        raise ValueError(f"Unknown compression format: {compression_format}")

    return PostingListIterator(
        docids=docids,
        tfs=tfs,
        skip_block_size=skip_block_size,
        ub_score=ub_score,
        idf=idf
    )

In [ ]:
%%writefile indexer.py
import gc
import heapq
import math
import pickle
import resource
import shutil
import struct
import sys
import time
from array import array
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

from text_processor import TextProcessor, parse_tsv_line
from postings_io import write_posting_list


@dataclass
class LexiconEntry:
    df: int                 # Document frequency df(t)
    cf: int                 # Collection frequency cf(t)
    offset: int             # Byte offset in postings file
    byte_length: int        # Length in bytes in postings file
    max_bm25: float         # Precomputed exact BM25 upper bound UB_t (for default k1, b)
    max_tf: int             # Max within-document term frequency (for dynamic UB with custom k1, b)
    min_dl: int             # Minimum length |d| among documents containing t


@dataclass
class IndexingReport:
    num_documents: int
    avg_doc_length: float
    total_tokens: int
    vocabulary_size: int
    skipped_lines: int
    empty_passages: int
    num_intermediate_runs: int
    run_generation_time_s: float
    merge_time_s: float
    total_indexing_time_s: float
    peak_rss_mb: float
    postings_size_bytes: int
    lexicon_size_bytes: int
    doc_table_size_bytes: int


def _get_peak_rss_mb() -> float:
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    divisor = (1024 ** 2) if sys.platform == "darwin" else 1024
    return usage / divisor


class _BlockRunReader:
    """Streaming reader over a sorted intermediate SPIMI block file"""
    def __init__(self, path: Path, block_id: int) -> None:
        self.path = path
        self.block_id = block_id
        self.f = open(path, "rb", buffering=1024 * 1024)
        self.current_term: str | None = None
        self.current_docids: bytes = b""
        self.current_tfs: bytes = b""
        self.current_df: int = 0

    def next_Entry(self) -> bool:
        header = self.f.read(2)
        if not header:
            self.current_term = None
            return False
        (term_len,) = struct.unpack("<H", header)
        term_bytes = self.f.read(term_len)
        self.current_term = term_bytes.decode("utf-8")
        (df,) = struct.unpack("<I", self.f.read(4))
        self.current_df = df
        byte_count = df * 4
        self.current_docids = self.f.read(byte_count)
        self.current_tfs = self.f.read(byte_count)
        return True

    def close(self) -> None:
        self.f.close()


class SPIMIIndexer:
    """
    Single pass in-memory indexing (SPIMI).
    Buffers postings in RAM up to max_buffer_postings, flushes sorted intermediate runs
    to disk, and merges them via a k-way min-heap into a single final inverted index
    """
    def __init__(
        self,
        text_processor: TextProcessor,
        max_buffer_postings: int = 15_000_000,
        compression_format: str = "none",
        bm25_k1: float = 0.9,
        bm25_b: float = 0.4,
        cleanup_intermediate_runs: bool = True
    ) -> None:
        self.tp = text_processor
        self.max_buffer_postings = max_buffer_postings
        self.compression_format = compression_format
        self.bm25_k1 = bm25_k1
        self.bm25_b = bm25_b
        self.cleanup_intermediate_runs = cleanup_intermediate_runs

    @staticmethod
    def _flush_block_to_disk(
        block_dict: dict[str, tuple[array, array]],
        run_path: Path
    ) -> None:
        """Sorts block dictionary keys lexicographically and writes a binary intermediate run"""
        tmp_path = run_path.with_suffix(".tmp")
        sorted_terms = sorted(block_dict.keys())
        with open(tmp_path, "wb", buffering=4 * 1024 * 1024) as f:
            for term in sorted_terms:
                docids_arr, tfs_arr = block_dict[term]
                term_bytes = term.encode("utf-8")
                df = len(docids_arr)
                f.write(struct.pack("<H", len(term_bytes)))
                f.write(term_bytes)
                f.write(struct.pack("<I", df))
                f.write(docids_arr.tobytes())
                f.write(tfs_arr.tobytes())
        tmp_path.replace(run_path)

    def build_index(
        self,
        collection_path: Path,
        output_dir: Path,
        show_progress: bool = True
    ) -> IndexingReport:
        output_dir.mkdir(parents=True, exist_ok=True)
        runs_dir = output_dir / "intermediate_runs"
        runs_dir.mkdir(parents=True, exist_ok=True)

        t_start = time.perf_counter()

        # ------ Phase 1: Stream passages and build sorted intermediate block runs ------
        t_phase1_start = time.perf_counter()
        doc_pids = array("I")
        doc_lengths = array("I")
        block_dict: dict[str, tuple[array, array]] = {}
        postings_in_buffer = 0
        run_paths: list[Path] = []

        skipped_lines = 0
        empty_passages = 0
        total_tokens = 0
        docid = 0

        with open(collection_path, "r", encoding="utf-8", errors="replace") as f:
            iterator = tqdm(f, desc="SPIMI phase 1 (parsing and blocks)", disable=not show_progress)
            for raw_line in iterator:
                pid, text = parse_tsv_line(raw_line)
                if pid is None:
                    skipped_lines += 1
                    continue

                tokens = self.tp.process(text)
                doc_len = len(tokens)
                if doc_len == 0:
                    empty_passages += 1

                doc_pids.append(pid)
                doc_lengths.append(doc_len)
                total_tokens += doc_len

                if doc_len > 0:
                    term_counts = Counter(tokens)
                    for term, tf in term_counts.items():
                        entry = block_dict.get(term)
                        if entry is None:
                            doc_arr = array("I", [docid])
                            tf_arr = array("I", [tf])
                            block_dict[term] = (doc_arr, tf_arr)
                        else:
                            entry[0].append(docid)
                            entry[1].append(tf)
                        postings_in_buffer += 1

                docid += 1

                if postings_in_buffer >= self.max_buffer_postings:
                    run_path = runs_dir / f"run_{len(run_paths):04d}.bin"
                    self._flush_block_to_disk(block_dict, run_path)
                    run_paths.append(run_path)
                    block_dict.clear()
                    postings_in_buffer = 0
                    gc.collect()

        # Flush remaining postings in buffer
        if block_dict:
            run_path = runs_dir / f"run_{len(run_paths):04d}.bin"
            self._flush_block_to_disk(block_dict, run_path)
            run_paths.append(run_path)
            block_dict.clear()
            postings_in_buffer = 0
            gc.collect()

        # Free stemmer cache after phase 1 to minimise memory footprint during Merge
        self.tp._stem_cache.clear()
        gc.collect()

        num_documents = docid
        avg_dl = (total_tokens / num_documents) if num_documents > 0 else 0.0
        run_gen_time = time.perf_counter() - t_phase1_start

        # Persist Document Table (doc_pids.npy and doc_lengths.npy)
        doc_pids_np = np.frombuffer(doc_pids, dtype=np.uint32).copy()
        doc_lengths_np = np.frombuffer(doc_lengths, dtype=np.uint32).copy()
        del doc_pids, doc_lengths
        gc.collect()

        pids_path = output_dir / "doc_pids.npy"
        lengths_path = output_dir / "doc_lengths.npy"
        np.save(pids_path, doc_pids_np)
        np.save(lengths_path, doc_lengths_np)
        del doc_pids_np
        gc.collect()

        # ------- Phase 2: k-way merge of intermediate runs into final index -------
        t_merge_start = time.perf_counter()
        postings_filename = "postings.txt" if self.compression_format == "text" else "postings.bin"
        postings_path = output_dir / postings_filename

        readers = [_BlockRunReader(p, idx) for idx, p in enumerate(run_paths)]
        heap: list[tuple[str, int]] = []
        for r in readers:
            if r.next_Entry():
                heapq.heappush(heap, (r.current_term, r.block_id))

        lexicon: dict[str, LexiconEntry] = {}
        k1 = self.bm25_k1
        b = self.bm25_b
        safe_avg_dl = avg_dl if avg_dl > 0.0 else 1.0

        with open(postings_path, "wb", buffering=8 * 1024 * 1024) as f_post:
            while heap:
                term, first_block_id = heapq.heappop(heap)
                doc_chunks = [readers[first_block_id].current_docids]
                tf_chunks = [readers[first_block_id].current_tfs]
                total_df = readers[first_block_id].current_df

                if readers[first_block_id].next_Entry():
                    heapq.heappush(heap, (readers[first_block_id].current_term, first_block_id))

                # Pop all other blocks sharing the exact same term in increasing block_id order
                while heap and heap[0][0] == term:
                    _, b_id = heapq.heappop(heap)
                    doc_chunks.append(readers[b_id].current_docids)
                    tf_chunks.append(readers[b_id].current_tfs)
                    total_df += readers[b_id].current_df
                    if readers[b_id].next_Entry():
                        heapq.heappush(heap, (readers[b_id].current_term, b_id))

                # Concatenate already-sorted disjoint docid ranges
                merged_docids = np.frombuffer(b"".join(doc_chunks), dtype=np.uint32)
                merged_tfs = np.frombuffer(b"".join(tf_chunks), dtype=np.uint32)

                cf = int(merged_tfs.sum())
                max_tf = int(merged_tfs.max())
                term_dls = doc_lengths_np[merged_docids]
                min_dl = int(term_dls.min())

                # Precompute exact BM25 upper bound
                idf = math.log(1.0 + (num_documents - total_df + 0.5) / (total_df + 0.5))
                denom = merged_tfs + k1 * (1.0 - b + b * (term_dls / safe_avg_dl))
                max_bm25 = float(np.max(idf * ((k1 + 1.0) * merged_tfs) / denom))

                offset, byte_len = write_posting_list(
                    f_post, term, merged_docids, merged_tfs, self.compression_format
                )

                lexicon[term] = LexiconEntry(
                    df=total_df,
                    cf=cf,
                    offset=offset,
                    byte_length=byte_len,
                    max_bm25=max_bm25,
                    max_tf=max_tf,
                    min_dl=min_dl
                )

        for r in readers:
            r.close()

        if self.cleanup_intermediate_runs:
            shutil.rmtree(runs_dir, ignore_errors=True)

        merge_time = time.perf_counter() - t_merge_start
        total_time = time.perf_counter() - t_start

        # Save lexicon and collection statistics
        lexicon_path = output_dir / "lexicon.pkl"
        with open(lexicon_path, "wb") as f_lex:
            pickle.dump(lexicon, f_lex, protocol=pickle.HIGHEST_PROTOCOL)

        stats_dict = {
            "num_documents": num_documents,
            "avg_doc_length": avg_dl,
            "total_tokens": total_tokens,
            "vocabulary_size": len(lexicon),
            "skipped_lines": skipped_lines,
            "empty_passages": empty_passages,
            "compression_format": self.compression_format,
            "use_stopwords": self.tp.use_stopwords,
            "use_stemming": self.tp.use_stemming,
            "unicode_normalisation": self.tp.unicode_normalisation,
            "bm25_k1": k1,
            "bm25_b": b,
        }
        stats_path = output_dir / "stats.pkl"
        with open(stats_path, "wb") as f_stats:
            pickle.dump(stats_dict, f_stats, protocol=pickle.HIGHEST_PROTOCOL)

        peak_rss = _get_peak_rss_mb()
        postings_bytes = postings_path.stat().st_size
        lexicon_bytes = lexicon_path.stat().st_size
        doc_table_bytes = pids_path.stat().st_size + lengths_path.stat().st_size

        return IndexingReport(
            num_documents=num_documents,
            avg_doc_length=avg_dl,
            total_tokens=total_tokens,
            vocabulary_size=len(lexicon),
            skipped_lines=skipped_lines,
            empty_passages=empty_passages,
            num_intermediate_runs=len(run_paths),
            run_generation_time_s=run_gen_time,
            merge_time_s=merge_time,
            total_indexing_time_s=total_time,
            peak_rss_mb=peak_rss,
            postings_size_bytes=postings_bytes,
            lexicon_size_bytes=lexicon_bytes,
            doc_table_size_bytes=doc_table_bytes
        )

In [ ]:
%%writefile query_processor.py
import heapq
import math
import mmap
import pickle
from dataclasses import dataclass
from pathlib import Path
import numpy as np

from text_processor import TextProcessor
from indexer import LexiconEntry
from postings_io import (
    SENTINEL_DOCID,
    PostingListIterator,
    load_posting_list_from_mmap
)


@dataclass
class QueryDiagnostics:
    """Stores telemetry for scored documents and accumulators"""
    query_length: int = 0
    scored_documents: int = 0
    num_accumulators: int = 0


class SearchEngine:
    """
    Disk-backed Search Engine supporting TF-IDF and BM25 over DAAT, TAAT, MaxScore, and WAND.
    Holds only lexicon, document table, and collection statistics in RAM
    """
    def __init__(
        self,
        index_dir: Path,
        bm25_k1: float | None = None,
        bm25_b: float | None = None,
        skip_block_size: int = 128,
        pruning_strategy: str = "none",
        traversal_strategy: str = "daat"
    ) -> None:
        self.index_dir = Path(index_dir)
        with open(self.index_dir / "stats.pkl", "rb") as f:
            self.stats = pickle.load(f)
        with open(self.index_dir / "lexicon.pkl", "rb") as f:
            self.lexicon: dict[str, LexiconEntry] = pickle.load(f)

        self.doc_pids: np.ndarray = np.load(self.index_dir / "doc_pids.npy")
        self.doc_lengths: np.ndarray = np.load(self.index_dir / "doc_lengths.npy")

        self.num_docs: int = int(self.stats["num_documents"])
        self.avg_dl: float = float(self.stats["avg_doc_length"])
        self.compression_format: str = self.stats["compression_format"]

        # TextProcessor configured identically to the index
        self.tp = TextProcessor(
            use_stopwords=self.stats["use_stopwords"],
            use_stemming=self.stats["use_stemming"],
            unicode_normalisation=self.stats.get("unicode_normalisation", "NFKC")
        )

        # Scoring and traversal parameters
        self.bm25_k1 = float(self.stats["bm25_k1"] if bm25_k1 is None else bm25_k1)
        self.bm25_b = float(self.stats["bm25_b"] if bm25_b is None else bm25_b)
        self.skip_block_size = skip_block_size
        self.pruning_strategy = pruning_strategy.lower()
        self.traversal_strategy = traversal_strategy.lower()

        # Precompute per document BM25 length normalisation denominator K(d) = k1 * (1 - b + b * |d| / |d|_avg)
        self._update_bm25_norm_table()

        # Open postings file via read-only memory map (OS reads pages on-demand per query)
        postings_fname = "postings.txt" if self.compression_format == "text" else "postings.bin"
        self._postings_file = open(self.index_dir / postings_fname, "rb")
        self._mmap = mmap.mmap(self._postings_file.fileno(), 0, access=mmap.ACCESS_READ)

        self.last_diagnostics = QueryDiagnostics()

    def _update_bm25_norm_table(self) -> None:
        safe_avg = self.avg_dl if self.avg_dl > 0.0 else 1.0
        self.bm25_K = (
            self.bm25_k1 * (1.0 - self.bm25_b + self.bm25_b * (self.doc_lengths.astype(np.float32) / safe_avg))
        )

    def set_bm25_params(self, k1: float, b: float) -> None:
        """Updates (k1, b) dynamically for Extension E7 grid search without reloading the index"""
        if k1 != self.bm25_k1 or b != self.bm25_b:
            self.bm25_k1 = k1
            self.bm25_b = b
            self._update_bm25_norm_table()

    def close(self) -> None:
        if self._mmap is not None:
            self._mmap.close()
            self._mmap = None
        if self._postings_file is not None:
            self._postings_file.close()
            self._postings_file = None

    def _compute_idf_and_ub(self, entry: LexiconEntry, scoring: str) -> tuple[float, float]:
        N = self.num_docs
        df = entry.df
        if scoring == "tfidf":
            idf = math.log(N / df)
            ub = (1.0 + math.log(entry.max_tf)) * idf
            return idf, ub
        elif scoring == "bm25":
            idf = math.log(1.0 + (N - df + 0.5) / (df + 0.5))
            # Use exact stored max_bm25 if using default parameters, else analytical tight bound
            if (
                abs(self.bm25_k1 - self.stats["bm25_k1"]) < 1e-9
                and abs(self.bm25_b - self.stats["bm25_b"]) < 1e-9
            ):
                return idf, entry.max_bm25
            safe_avg = self.avg_dl if self.avg_dl > 0.0 else 1.0
            min_K = self.bm25_k1 * (1.0 - self.bm25_b + self.bm25_b * (entry.min_dl / safe_avg))
            ub = idf * ((self.bm25_k1 + 1.0) * entry.max_tf) / (entry.max_tf + min_K)
            return idf, ub
        else:
            raise ValueError(f"Unknown scoring function: {scoring}")

    def _open_query_cursors(
        self,
        terms: list[str],
        scoring: str
    ) -> list[PostingListIterator]:
        cursors: list[PostingListIterator] = []
        for t in terms:
            entry = self.lexicon[t]
            idf, ub = self._compute_idf_and_ub(entry, scoring)
            cursors.append(
                load_posting_list_from_mmap(
                    mmap_buf=self._mmap,
                    offset=entry.offset,
                    byte_length=entry.byte_length,
                    df=entry.df,
                    compression_format=self.compression_format,
                    skip_block_size=self.skip_block_size,
                    ub_score=ub,
                    idf=idf
                )
            )
        return cursors

    def _finalize_heap(self, heap: list[tuple[float, int]]) -> list[tuple[int, float]]:
        """
        Sorts the min-heap of (score, -docid) in decreasing order of score,
        breaking ties by increasing docid, and maps internal docid -> external pid
        """
        heap.sort(key=lambda item: (-item[0], -item[1]))
        pids = self.doc_pids
        return [(int(pids[-neg_docid]), float(score)) for score, neg_docid in heap]


    # Public Search API
    def search(
        self,
        query: str,
        k: int = 1000,
        mode: str = "or",
        scoring: str = "bm25",
        traversal: str | None = None,
        pruning: str | None = None
    ) -> list[tuple[int, float]]:
        mode = mode.lower()
        scoring = scoring.lower()
        trav = (traversal or self.traversal_strategy).lower()
        prun = (pruning or self.pruning_strategy).lower()

        if mode not in ("and", "or"):
            raise ValueError(f"mode must be 'and' or 'or', got '{mode}'")
        if scoring not in ("tfidf", "bm25"):
            raise ValueError(f"scoring must be 'tfidf' or 'bm25', got '{scoring}'")
        if k <= 0:
            self.last_diagnostics = QueryDiagnostics()
            return []

        # Preprocess query and deduplicate terms (each distinct term counted once)
        raw_tokens = self.tp.process(query)
        unique_terms = list(dict.fromkeys(raw_tokens))
        q_len = len(unique_terms)

        if q_len == 0:
            self.last_diagnostics = QueryDiagnostics(query_length=0)
            return []

        if mode == "and":
            # In AND mode, if any query term is absent from the lexicon, result is empty
            for t in unique_terms:
                if t not in self.lexicon:
                    self.last_diagnostics = QueryDiagnostics(query_length=q_len)
                    return []
            present_terms = unique_terms
        else:
            # In OR mode, missing terms are ignored
            present_terms = [t for t in unique_terms if t in self.lexicon]
            if not present_terms:
                self.last_diagnostics = QueryDiagnostics(query_length=q_len)
                return []

        cursors = self._open_query_cursors(present_terms, scoring)

        if trav == "taat":
            return self._search_taat(cursors, k, mode, scoring, q_len)

        # Default: DAAT traversal
        if mode == "and":
            return self._daat_and(cursors, k, scoring, q_len)
        else:
            if prun == "maxscore":
                return self._daat_or_maxscore(cursors, k, scoring, q_len)
            elif prun == "wand":
                return self._daat_or_wand(cursors, k, scoring, q_len)
            else:
                return self._daat_or_exhaustive(cursors, k, scoring, q_len)


    # DAAT Conjunctive (AND) Processing (with Skip Pointers via next_geq)
    def _daat_and(
        self,
        cursors: list[PostingListIterator],
        k: int,
        scoring: str,
        q_len: int
    ) -> list[tuple[int, float]]:
        # Sort posting lists by increasing length |L_1| <= |L_2| <= ... <= |L_m|
        cursors.sort(key=lambda c: c.n)
        lead = cursors[0]
        others = cursors[1:]

        is_bm25 = (scoring == "bm25")
        k1_plus_1 = self.bm25_k1 + 1.0
        bm25_K = self.bm25_K
        log_fn = math.log

        heap: list[tuple[float, int]] = []
        scored_docs = 0

        cand = lead.docid
        while cand != SENTINEL_DOCID:
            match = True
            next_target = cand
            for c in others:
                d = c.next_geq(cand)
                if d == SENTINEL_DOCID:
                    self.last_diagnostics = QueryDiagnostics(q_len, scored_docs, 0)
                    return self._finalize_heap(heap)
                if d > cand:
                    match = False
                    next_target = d
                    break

            if match:
                scored_docs += 1
                score = 0.0
                if is_bm25:
                    K_d = float(bm25_K[cand])
                    for c in cursors:
                        tf = c.tf
                        score += c.idf * (k1_plus_1 * tf) / (tf + K_d)
                else:
                    for c in cursors:
                        score += (1.0 + log_fn(c.tf)) * c.idf

                item = (score, -cand)
                if len(heap) < k:
                    heapq.heappush(heap, item)
                elif item > heap[0]:
                    heapq.heapreplace(heap, item)

                cand = lead.advance()
            else:
                cand = lead.next_geq(next_target)

        self.last_diagnostics = QueryDiagnostics(q_len, scored_docs, 0)
        return self._finalize_heap(heap)


    # DAAT Disjunctive (OR) Exhaustive Processing
    def _daat_or_exhaustive(
        self,
        cursors: list[PostingListIterator],
        k: int,
        scoring: str,
        q_len: int
    ) -> list[tuple[int, float]]:
        is_bm25 = (scoring == "bm25")
        k1_plus_1 = self.bm25_k1 + 1.0
        bm25_K = self.bm25_K
        log_fn = math.log
        m = len(cursors)

        heap: list[tuple[float, int]] = []
        scored_docs = 0

        while True:
            # Find minimum current docid across active cursors
            min_doc = SENTINEL_DOCID
            for i in range(m):
                d = cursors[i].docid
                if d < min_doc:
                    min_doc = d

            if min_doc == SENTINEL_DOCID:
                break

            scored_docs += 1
            score = 0.0
            if is_bm25:
                K_d = float(bm25_K[min_doc])
                for i in range(m):
                    c = cursors[i]
                    if c.docid == min_doc:
                        tf = c.tf
                        score += c.idf * (k1_plus_1 * tf) / (tf + K_d)
                        c.advance()
            else:
                for i in range(m):
                    c = cursors[i]
                    if c.docid == min_doc:
                        score += (1.0 + log_fn(c.tf)) * c.idf
                        c.advance()

            item = (score, -min_doc)
            if len(heap) < k:
                heapq.heappush(heap, item)
            elif item > heap[0]:
                heapq.heapreplace(heap, item)

        self.last_diagnostics = QueryDiagnostics(q_len, scored_docs, 0)
        return self._finalize_heap(heap)


    # Extension E2: MaxScore Dynamic Pruning
    def _daat_or_maxscore(
        self,
        cursors: list[PostingListIterator],
        k: int,
        scoring: str,
        q_len: int
    ) -> list[tuple[int, float]]:
        # Sort cursors by increasing upper bound: UB_1 <= UB_2 <= ... <= UB_m
        cursors.sort(key=lambda c: c.ub_score)
        m = len(cursors)

        # Prefix sums of upper bounds: S[i] = sum_{j=0}^{i-1} UB_j
        prefix_ub = [0.0] * (m + 1)
        for i in range(m):
            prefix_ub[i + 1] = prefix_ub[i] + cursors[i].ub_score

        is_bm25 = (scoring == "bm25")
        k1_plus_1 = self.bm25_k1 + 1.0
        bm25_K = self.bm25_K
        log_fn = math.log

        heap: list[tuple[float, int]] = []
        theta = 0.0
        p = 0  # Non-essential boundary: cursors[0..p-1] are non-essential, cursors[p..m-1] are essential
        scored_docs = 0
        eps = 1e-9

        while p < m:
            # Find minimum docid among essential cursors (p .. m-1)
            cand = SENTINEL_DOCID
            for i in range(p, m):
                d = cursors[i].docid
                if d < cand:
                    cand = d

            if cand == SENTINEL_DOCID:
                break

            scored_docs += 1
            score = 0.0
            K_d = float(bm25_K[cand]) if is_bm25 else 0.0

            # Accumulate score from essential terms containing cand
            for i in range(p, m):
                c = cursors[i]
                if c.docid == cand:
                    if is_bm25:
                        tf = c.tf
                        score += c.idf * (k1_plus_1 * tf) / (tf + K_d)
                    else:
                        score += (1.0 + log_fn(c.tf)) * c.idf
                    c.advance()

            # Check non-essential terms from highest UB down to 0, stopping early if bound <= theta
            if score + prefix_ub[p] >= theta - eps:
                for i in range(p - 1, -1, -1):
                    c = cursors[i]
                    d = c.next_geq(cand)
                    if d == cand:
                        if is_bm25:
                            tf = c.tf
                            score += c.idf * (k1_plus_1 * tf) / (tf + K_d)
                        else:
                            score += (1.0 + log_fn(c.tf)) * c.idf
                    if score + prefix_ub[i] < theta - eps:
                        break

            item = (score, -cand)
            if len(heap) < k:
                heapq.heappush(heap, item)
                if len(heap) == k:
                    theta = heap[0][0]
            elif item > heap[0]:
                heapq.heapreplace(heap, item)
                theta = heap[0][0]

            # Update non-essential partition boundary p while S[p+1] < theta
            while p < m and prefix_ub[p + 1] < theta - eps:
                p += 1

        self.last_diagnostics = QueryDiagnostics(q_len, scored_docs, 0)
        return self._finalize_heap(heap)


    # Extension E2: WAND Dynamic Pruning
    def _daat_or_wand(
        self,
        cursors: list[PostingListIterator],
        k: int,
        scoring: str,
        q_len: int
    ) -> list[tuple[int, float]]:
        is_bm25 = (scoring == "bm25")
        k1_plus_1 = self.bm25_k1 + 1.0
        bm25_K = self.bm25_K
        log_fn = math.log
        m = len(cursors)

        heap: list[tuple[float, int]] = []
        theta = 0.0
        scored_docs = 0
        eps = 1e-9

        while True:
            # Sort active cursors by current docid
            cursors.sort(key=lambda c: c.docid)
            if cursors[0].docid == SENTINEL_DOCID:
                break

            # Find pivot p: smallest index such that sum_{i=0}^p UB_i >= theta
            acc_ub = 0.0
            pivot_idx = -1
            for i in range(m):
                if cursors[i].docid == SENTINEL_DOCID:
                    break
                acc_ub += cursors[i].ub_score
                if acc_ub >= theta - eps:
                    pivot_idx = i
                    break

            if pivot_idx == -1:
                break

            d_pivot = cursors[pivot_idx].docid
            if d_pivot == SENTINEL_DOCID:
                break

            # Check if all cursors up to pivot_idx are aligned at d_pivot
            if cursors[0].docid == d_pivot:
                scored_docs += 1
                score = 0.0
                K_d = float(bm25_K[d_pivot]) if is_bm25 else 0.0
                i = 0
                while i < m and cursors[i].docid == d_pivot:
                    c = cursors[i]
                    if is_bm25:
                        tf = c.tf
                        score += c.idf * (k1_plus_1 * tf) / (tf + K_d)
                    else:
                        score += (1.0 + log_fn(c.tf)) * c.idf
                    c.advance()
                    i += 1

                item = (score, -d_pivot)
                if len(heap) < k:
                    heapq.heappush(heap, item)
                    if len(heap) == k:
                        theta = heap[0][0]
                elif item > heap[0]:
                    heapq.heapreplace(heap, item)
                    theta = heap[0][0]
            else:
                # Advance leading cursor(s) before d_pivot up to d_pivot via next_geq
                cursors[0].next_geq(d_pivot)

        self.last_diagnostics = QueryDiagnostics(q_len, scored_docs, 0)
        return self._finalize_heap(heap)


    # Term-At-A-Time (TAAT) Processing
    def _search_taat(
        self,
        cursors: list[PostingListIterator],
        k: int,
        mode: str,
        scoring: str,
        q_len: int
    ) -> list[tuple[int, float]]:
        is_bm25 = (scoring == "bm25")
        k1_plus_1 = self.bm25_k1 + 1.0
        bm25_K = self.bm25_K

        if mode == "or":
            acc: dict[int, float] = {}
            for c in cursors:
                docs = c.docids
                tfs = c.tfs
                idf = c.idf
                if is_bm25:
                    scores = idf * (k1_plus_1 * tfs) / (tfs + bm25_K[docs])
                else:
                    scores = (1.0 + np.log(tfs)) * idf

                for d_val, s_val in zip(docs.tolist(), scores.tolist()):
                    acc[d_val] = acc.get(d_val, 0.0) + s_val

            num_acc = len(acc)
        else:
            # TAAT Conjunctive (AND): sort by increasing df and intersect accumulator maps
            cursors.sort(key=lambda c: c.n)
            acc = {}
            for idx, c in enumerate(cursors):
                docs = c.docids
                tfs = c.tfs
                idf = c.idf
                if idx == 0:
                    if is_bm25:
                        scores = idf * (k1_plus_1 * tfs) / (tfs + bm25_K[docs])
                    else:
                        scores = (1.0 + np.log(tfs)) * idf
                    acc = dict(zip(docs.tolist(), scores.tolist()))
                else:
                    next_acc: dict[int, float] = {}
                    for d_val, tf_val in zip(docs.tolist(), tfs.tolist()):
                        prev_score = acc.get(d_val)
                        if prev_score is not None:
                            if is_bm25:
                                contrib = idf * (k1_plus_1 * tf_val) / (tf_val + float(bm25_K[d_val]))
                            else:
                                contrib = (1.0 + math.log(tf_val)) * idf
                            next_acc[d_val] = prev_score + contrib
                    acc = next_acc
                if not acc:
                    break
            num_acc = len(cursors[0].docids) if cursors else 0

        heap: list[tuple[float, int]] = []
        for docid, score in acc.items():
            item = (score, -docid)
            if len(heap) < k:
                heapq.heappush(heap, item)
            elif item > heap[0]:
                heapq.heapreplace(heap, item)

        self.last_diagnostics = QueryDiagnostics(
            query_length=q_len,
            scored_documents=len(acc),
            num_accumulators=num_acc
        )
        return self._finalize_heap(heap)

In [ ]:
from text_processor import TextProcessor, parse_tsv_line, ENGLISH_STOP_WORDS
from postings_io import PostingListIterator, write_posting_list, load_posting_list_from_mmap
from indexer import SPIMIIndexer, LexiconEntry, IndexingReport
from query_processor import SearchEngine, QueryDiagnostics

# Global reference to the active SearchEngine instance
ACTIVE_ENGINE: SearchEngine | None = None

def set_active_engine(engine: SearchEngine) -> None:
    """Binds a SearchEngine instance to the top-level search function"""
    global ACTIVE_ENGINE
    ACTIVE_ENGINE = engine

def search(query: str, k: int, mode: str, scoring: str) -> list[tuple[int, float]]:
    """
    Top-level search interface
    Parameters:
      - query   : raw query string
      - k       : maximum number of documents to return
      - mode    : "and" or "or"
      - scoring : "tfidf" or "bm25"
    Returns:
      - list of at most k pairs (pid, score) in decreasing order of score,
        with score ties broken by increasing docid
    """
    if ACTIVE_ENGINE is None:
        raise RuntimeError("No active SearchEngine loaded. Call set_active_engine(engine) first")
    return ACTIVE_ENGINE.search(query=query, k=k, mode=mode, scoring=scoring)

print("Section 3 modules compiled and imported successfully")